# CodeRAG-Bench evaluation: HumanEval/MBPP

[CodeRAG-Bench](https://github.com/code-rag-bench/code-rag-bench) is a
published benchmark for retrieval-augmented code generation. It pairs coding
tasks (HumanEval, MBPP, DS-1000, ODEX, RepoEval, SWE-bench, ...) with
retrieval corpora, and scores retrieval with the same information-retrieval
metrics BEIR uses: NDCG, MRR, Recall, and Precision.

This notebook runs the evaluation protocol against QAChat's own retrieval
strategies. It starts with the two smallest, fully self-contained tasks CodeRAG-Bench calls
**canonical**: HumanEval and MBPP, document-to-code retrieval. The corpus is every problem's own canonical solution, one per problem, taken directly from the benchmark's dataset (no external documents, no unrelated code). The query is a coding problem's prompt/description, and retrieval must find that same problem's own solution among all the others' solutions in the corpus.

Each task is stored in [BEIR](https://github.com/beir-cellar/beir) format:
- `corpus`: `{doc_id: {"title": ..., "text": ...}}` — one entry per problem's solution.
- `queries`: `{query_id: text}` — one entry per problem's prompt/description.
- `qrels`: `{query_id: {doc_id: relevance}}` — which corpus document(s) count as correct for each query, and how relevant (here always exactly one document, relevance 1).

Example:

**Query** (`HumanEval/0_doc`):
```python
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
```

**Gold corpus document** (`HumanEval/0_code`, `qrels["HumanEval/0_doc"] == {"HumanEval/0_code": 1}`): that same text, followed by its canonical solution —

```python
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False
```

A perfect retriever ranks `HumanEval/0_code` first among every other HumanEval solution in the corpus for this query.

**Important:** Since the original evaluation keeps every document whole — one document is one retrievable unit, never split — but QAChat's real pipeline chunks documents before indexing them, a single corpus document can become several chunks with new, content-based identifiers. The original `qrels` are keyed by whole-document id, so as soon as a gold document splits, its id no longer exists in the index and `evaluate_retrieval` would score that query as a miss even when retrieval found the right document. We fix this by propagating each document's relevance to every chunk produced from it, so a hit on *any* chunk of the gold document still counts.

**Scope: retrieval only.** This notebook scores `_retrieve()` -- the retrieval
half of `RAGSession.ask()` -- against BEIR's ranking metrics
(NDCG/MRR/Recall). It never runs `_answer_prompt()` or the chat model,
so a query where the gold chunk ranks #1 means retrieval found the
right material, not that a model would necessarily answer correctly
from it.

## 1. Prepare QAChat

In [1]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [3]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Benchmark-only dependencies

In [4]:
try:
    import datasets
    import pytrec_eval
except ImportError as error:
    raise RuntimeError(
        "Benchmark dependencies are missing. From the QAChat root, run:\n"
        "    uv sync --extra mlx --extra torch --extra bench\n"
        "(always pass every extra together -- a bare '--extra bench' or\n"
        "'--extra mlx --extra torch' will each uninstall what the other\n"
        "installed) then select this notebook's kernel again."
    ) from error

print(f"datasets {datasets.__version__}")


/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets 5.0.1


## 3. Build the HumanEval and MBPP retrieval tasks

CodeRAG-Bench stores each task in [BEIR](https://github.com/beir-cellar/beir)
format: a `corpus` of `{id: {"title", "text"}}`, a set of `queries` of
`{id: text}`, and `qrels` mapping each query id to the corpus ids that
answer it, with a relevance score.

`BenchmarkTask` is that same shape as one dataclass, so it can be built
once per task and reused for every retrieval channel.


In [5]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class BenchmarkTask:
    """One CodeRAG-Bench retrieval task in BEIR format.

    Attributes
    ----------
    name: str
        Short task name, used in result tables.
    corpus: dict[str, dict[str, str]]
        Document id -> {"title": ..., "text": ...}.
    queries: dict[str, str]
        Query id -> query text.
    qrels: dict[str, dict[str, int]]
        Query id -> {document id: relevance score}.

    """

    name: str
    corpus: dict[str, dict[str, str]]
    queries: dict[str, str]
    qrels: dict[str, dict[str, int]]


In [6]:
def build_humaneval_task() -> BenchmarkTask:
    """Build the HumanEval document-to-code retrieval task.

    Mirrors CodeRAG-Bench's ``retrieval/create/humaneval.py``: the query is
    a problem's prompt, and the matching corpus document is that same
    prompt followed by its canonical solution.
    """

    rows = datasets.load_dataset("openai/openai_humaneval")["test"]

    corpus: dict[str, dict[str, str]] = {}
    queries: dict[str, str] = {}
    qrels: dict[str, dict[str, int]] = {}

    for row in rows:
        doc_id = f"{row['task_id']}_doc"
        code_id = f"{row['task_id']}_code"
        queries[doc_id] = row["prompt"]
        corpus[code_id] = {
            "title": row["entry_point"],
            "text": row["prompt"] + "\n" + row["canonical_solution"],
        }
        qrels[doc_id] = {code_id: 1}

    return BenchmarkTask("humaneval", corpus, queries, qrels)


humaneval_task = build_humaneval_task()
print(
    f"{humaneval_task.name}: {len(humaneval_task.queries)} queries, "
    f"{len(humaneval_task.corpus)} corpus documents"
)


humaneval: 164 queries, 164 corpus documents


In [7]:
def _mbpp_function_name(code_text: str) -> str:
    """Return the function name declared in one MBPP solution."""

    for line in code_text.split("\n"):
        stripped = line.lstrip()
        if stripped.startswith("def "):
            return stripped[4:].split("(")[0]
    return ""


def build_mbpp_task() -> BenchmarkTask:
    """Build the MBPP document-to-code retrieval task.

    Mirrors CodeRAG-Bench's ``retrieval/create/mbpp.py``. Only the test
    split is used here to keep this first pass fast; the published
    benchmark also indexes the train and validation splits into the
    corpus, which a later pass can add.
    """

    rows = datasets.load_dataset("google-research-datasets/mbpp")["test"]

    corpus: dict[str, dict[str, str]] = {}
    queries: dict[str, str] = {}
    qrels: dict[str, dict[str, int]] = {}

    for row in rows:
        doc_id = f"{row['task_id']}_doc"
        code_id = f"{row['task_id']}_code"
        queries[doc_id] = row["text"]
        corpus[code_id] = {
            "title": _mbpp_function_name(row["code"]),
            "text": "# " + row["text"] + "\n" + row["code"],
        }
        qrels[doc_id] = {code_id: 1}

    return BenchmarkTask("mbpp", corpus, queries, qrels)


mbpp_task = build_mbpp_task()
print(
    f"{mbpp_task.name}: {len(mbpp_task.queries)} queries, "
    f"{len(mbpp_task.corpus)} corpus documents"
)


mbpp: 500 queries, 500 corpus documents


In [8]:
tasks = (humaneval_task, mbpp_task)


## 4. Load the embedding model

Loaded before real chunking, because `chunk_source_units` (section 5) needs a real token counter from the embedding model.


In [9]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}


In [10]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")


Backend: mlx
Model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


In [11]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2507.29it/s]


Loaded on: metal


In [12]:
from qachat.query_formats import format_qwen3_query

CODE_RETRIEVAL_INSTRUCTION = (
    "Given a coding problem description, retrieve the source code that "
    "solves it."
)


def format_query(question: str) -> str:
    """Format one benchmark query for the Qwen3 embedding model."""

    return format_qwen3_query(
        question=question,
        instruction=CODE_RETRIEVAL_INSTRUCTION,
    )


## 5. Turn the corpus into a real QAChat project

`RAGSession._prepare_source`  scans a project folder on disk. To exercise that same path, every corpus
document is written out as its own `.py` file under
`sandbox/humaneval_mbpp_benchmark/repository/<task_name>/`, named after
its corpus id so a chunk's file path can be traced straight back to the
document it came from. `sandbox/humaneval_mbpp_benchmark/profile.ini`
is then opened as a real `QAProject`.


In [13]:
def _doc_filename(doc_id: str) -> str:
    """Turn one corpus id into a safe, unique .py filename."""

    return doc_id.replace("/", "__") + ".py"


project_root = root / "sandbox" / "humaneval_mbpp_benchmark"
profile_path = project_root / "profile.ini"
if not profile_path.exists():
    raise RuntimeError(
        f"Missing project profile: {profile_path}. This notebook expects "
        "the profile checked into the repository at that path."
    )

path_to_doc_id: dict[str, str] = {}

for task in tasks:
    task_dir = project_root / "repository" / task.name
    task_dir.mkdir(parents=True, exist_ok=True)
    for doc_id, doc in task.corpus.items():
        file_path = task_dir / _doc_filename(doc_id)
        file_path.write_text(doc["text"])
        relative_path = file_path.relative_to(project_root / "repository")
        path_to_doc_id[str(relative_path)] = doc_id

print(f"Wrote {len(path_to_doc_id)} corpus documents under {project_root / 'repository'}")


Wrote 664 corpus documents under /Users/cmanzo/Documents/GitHub/QAChat/sandbox/humaneval_mbpp_benchmark/repository


In [14]:
from qachat.project import open_project

project = open_project(project_root)
profile = project.profile

print(f"Project source: {project.source}")
print(
    f"Chunking: target={profile.target_tokens} max={profile.max_tokens} "
    f"overlap={profile.overlap_tokens}"
)


Project source: /Users/cmanzo/Documents/GitHub/QAChat/sandbox/humaneval_mbpp_benchmark/repository
Chunking: target=300 max=400 overlap=40


In [15]:
from qachat.chunking import Chunk, chunk_source_units
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files

repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
corpus_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

print(f"{len(source_files)} files -> {len(source_units)} source units -> {len(corpus_chunks)} chunks")


664 files -> 1217 source units -> 1223 chunks


<unknown>:4: SyntaxWarning: invalid escape sequence '\W'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:5: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\*'


## 6. Propagate document-level qrels to chunk level

Now that real chunking has run, check the assumption the fix above
depends on: how many corpus documents actually split into more than one
chunk?


In [16]:
chunks_by_doc_id: dict[str, list[Chunk]] = {}
for chunk in corpus_chunks:
    doc_id = path_to_doc_id.get(str(chunk.path))
    if doc_id is None:
        continue
    chunks_by_doc_id.setdefault(doc_id, []).append(chunk)

split_counts: dict[str, int] = {}
for task in tasks:
    doc_ids = list(task.corpus)
    counts = [len(chunks_by_doc_id.get(doc_id, [])) for doc_id in doc_ids]
    split = sum(1 for count in counts if count > 1)
    empty = sum(1 for count in counts if count == 0)
    split_counts[task.name] = split
    print(
        f"{task.name}: {len(doc_ids)} documents, {split} split into >1 chunk, "
        f"{empty} produced 0 chunks (max chunks for one document: "
        f"{max(counts) if counts else 0})"
    )


def propagate_qrels(task: BenchmarkTask) -> dict[str, dict[str, int]]:
    """Give every chunk of a relevant document that document's relevance."""

    qrels_chunks: dict[str, dict[str, int]] = {}
    for query_id, doc_scores in task.qrels.items():
        chunk_scores: dict[str, int] = {}
        for doc_id, relevance in doc_scores.items():
            for chunk in chunks_by_doc_id.get(doc_id, ()):
                chunk_scores[chunk.identifier] = relevance
        qrels_chunks[query_id] = chunk_scores
    return qrels_chunks


qrels_chunks: dict[str, dict[str, dict[str, int]]] = {
    task.name: propagate_qrels(task) for task in tasks
}

missing = {
    task.name: sum(1 for scores in qrels_chunks[task.name].values() if not scores)
    for task in tasks
}
print(f"Queries left with zero relevant chunks after propagation: {missing}")


humaneval: 164 documents, 31 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 3)
mbpp: 500 documents, 500 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 5)
Queries left with zero relevant chunks after propagation: {'humaneval': 0, 'mbpp': 0}


### Verify: every chunk of a gold document is actually covered

Not a scoring change -- a check that the propagation above did what it
claims. For every query, every chunk belonging to that query's gold
document should now carry the document's relevance in `qrels_chunks`,
whatever rank that chunk comes back at. If this ever fails, it means
`path_to_doc_id` silently missed a chunk (a real bug), not a scoring
gap.

In [17]:
mismatches = []

for task in tasks:
    for query_id, doc_scores in task.qrels.items():
        for doc_id, relevance in doc_scores.items():
            expected_chunk_ids = {chunk.identifier for chunk in chunks_by_doc_id.get(doc_id, ())}
            actual_chunk_ids = {
                chunk_id
                for chunk_id, score in qrels_chunks[task.name][query_id].items()
                if score == relevance
            }
            if expected_chunk_ids != actual_chunk_ids:
                mismatches.append((task.name, query_id, doc_id, expected_chunk_ids, actual_chunk_ids))

if mismatches:
    raise AssertionError(
        f"{len(mismatches)} document(s) have chunks missing from qrels_chunks "
        f"after propagation -- first: {mismatches[0]}"
    )

print("Every chunk of every gold document is present in qrels_chunks with the document's relevance.")

Every chunk of every gold document is present in qrels_chunks with the document's relevance.


### Example: one document through real chunking and propagation

Picking a document is data-driven, not staged: prefer one that actually
split into multiple chunks, since that's the case the propagation logic
exists for; fall back to an unsplit one (labelled as such) if none did.


In [18]:
example_task = mbpp_task
example_doc_id = next(
    (
        doc_id
        for doc_id in example_task.corpus
        if len(chunks_by_doc_id.get(doc_id, [])) > 1
    ),
    None,
)
split_example = example_doc_id is not None
if example_doc_id is None:
    example_doc_id = next(iter(example_task.corpus))

example_query_id = next(
    query_id
    for query_id, doc_scores in example_task.qrels.items()
    if example_doc_id in doc_scores
)
example_chunks = chunks_by_doc_id.get(example_doc_id, [])

print(f"Document: {example_doc_id} ({'split' if split_example else 'not split'} by real chunking)")
print(f"Query {example_query_id}: {example_task.queries[example_query_id]!r}")
print(f"\nOriginal (1x0) qrel: {{{example_doc_id!r}: 1}}")
print(f"\nReal chunking produced {len(example_chunks)} chunk(s) for this document:")
for chunk in example_chunks:
    print(f"  {chunk.identifier}  lines {chunk.start_line}-{chunk.end_line}  kind={chunk.kind}  symbol={chunk.symbol}")

print(f"\nPropagated qrel for query {example_query_id}:")
print({chunk.identifier: 1 for chunk in example_chunks})


Document: 11_code (split by real chunking)
Query 11_doc: 'Write a python function to remove first and last occurrence of a given character from the string.'

Original (1x0) qrel: {'11_code': 1}

Real chunking produced 2 chunk(s) for this document:
  ccc2e52c6af271f0  lines 1-1  kind=python_module_context  symbol=11_code
  47a779410abd903e  lines 2-11  kind=python_function  symbol=remove_Occ

Propagated qrel for query 11_doc:
{'ccc2e52c6af271f0': 1, '47a779410abd903e': 1}


## 7. Build the code-aware retrieval channel

HumanEval's query *is* the function's own signature and docstring (it
literally contains the function name, since the query and the gold
solution share the same `prompt` text) -- `code_search`'s exact
identifier matching should find the gold chunk very often. MBPP's query
is a free-form natural-language task description, written independently
of the function name it goes with -- `code_search` should do
meaningfully worse there.


In [19]:
from qachat.retrievals import build_code_graph

code_graph = build_code_graph(source_files, source_units)

print(f"Code graph: {len(code_graph.entities)} entities, {len(code_graph.relations)} relations")


Code graph: 1886 entities, 1996 relations


<unknown>:4: SyntaxWarning: invalid escape sequence '\W'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:5: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:4: SyntaxWarning: invalid escape sequence '\*'


## 8. Run retrieval

Every channel's retrieval function is defined here, against the real
`corpus_chunks` built in section 5. Real QAChat never calls
`semantic_search` on its own -- `RAGSession._retrieve` always fuses
`semantic_search` + `lexical_search` + `code_search` through
`hybrid_search`. So `run_hybrid_retrieval(..., with_code=True)` is the
one call that matches the real pipeline; `run_semantic_retrieval`,
`run_lexical_retrieval`, `run_code_retrieval`, and
`run_hybrid_retrieval(..., with_code=False)` are ablations defined here
to explain *why* the real pipeline scores what it scores in section 10,
not independent candidates.

Each task's `semantic_index`/`lexical_index` are built exactly once,
right after these functions, and passed into every call below instead
of being rebuilt per call -- building an index means embedding the whole
corpus, so rebuilding it inside every `run_*_retrieval` call (once for
`hybrid+code`, again for each ablation, again for the worked example)
was re-embedding the same chunks up to ten times over.


In [20]:
from qachat import (
    build_lexical_index,
    build_semantic_index,
    code_search,
    hybrid_search,
    lexical_search,
    semantic_search,
)


def task_chunks(task: BenchmarkTask) -> list[Chunk]:
    """Return the real chunks belonging to one task's documents."""

    return [
        chunk
        for doc_id in task.corpus
        for chunk in chunks_by_doc_id.get(doc_id, ())
    ]


def run_semantic_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    semantic_index,
    embedding_session: EmbeddingSession,
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's semantic search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    for query_id, query_text in task.queries.items():
        ranked = semantic_search(
            semantic_index,
            format_query(query_text),
            embedding_session,
            count=count,
        )
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_lexical_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    lexical_index,
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's lexical search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    for query_id, query_text in task.queries.items():
        ranked = lexical_search(lexical_index, query_text, count=count)
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_code_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's code-aware search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    for query_id, query_text in task.queries.items():
        ranked = code_search(code_graph, chunks, query_text, count=count)
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_hybrid_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    semantic_index,
    lexical_index,
    embedding_session: EmbeddingSession,
    *,
    count: int = 100,
    with_code: bool = False,
) -> dict[str, dict[str, float]]:
    """Fuse semantic, lexical, and (optionally) code rankings."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    for query_id, query_text in task.queries.items():
        semantic_ranked = semantic_search(
            semantic_index,
            format_query(query_text),
            embedding_session,
            count=count,
        )
        lexical_ranked = lexical_search(
            lexical_index, query_text, count=count
        )
        code_ranked = (
            code_search(code_graph, chunks, query_text, count=count)
            if with_code
            else ()
        )
        fused = hybrid_search(
            semantic=semantic_ranked,
            lexical=lexical_ranked,
            code=code_ranked,
            count=count,
        )
        results[query_id] = {
            result.chunk.identifier: result.score for result in fused
        }
    return results


task_indexes: dict[str, tuple[list[Chunk], object, object]] = {}
for task in tasks:
    chunks = task_chunks(task)
    semantic_index = build_semantic_index(chunks, embedding_session)
    lexical_index = build_lexical_index(chunks)
    task_indexes[task.name] = (chunks, semantic_index, lexical_index)
    print(f"{task.name}: indexed {len(chunks)} chunks")


humaneval: indexed 196 chunks
mbpp: indexed 1027 chunks


## 9. Score the real pipeline: hybrid+code

`evaluate_retrieval` reimplements BEIR's `EvaluateRetrieval.evaluate`
and its custom MRR metric directly against `pytrec_eval`. Scored first:
`run_hybrid_retrieval(..., with_code=True)` -- the one call that
actually matches `RAGSession._retrieve`, since QAChat's real pipeline
never runs semantic search (or any single channel) on its own.
Everything else is an ablation, scored in section 10 to explain this
result, not to compete with it. Raw per-query results are kept in
`task_results`, not just the aggregate scores, so section 10's ablations
and the worked example can reuse them instead of retrieving again.


In [21]:
K_VALUES = [1, 3, 5, 10, 100]


def evaluate_retrieval(
    qrels: dict[str, dict[str, int]],
    results: dict[str, dict[str, float]],
    k_values: list[int] = K_VALUES,
) -> dict[str, dict[str, float]]:
    """Score ranked results the same way CodeRAG-Bench and BEIR do."""

    ndcg = {f"NDCG@{k}": 0.0 for k in k_values}
    map_ = {f"MAP@{k}": 0.0 for k in k_values}
    recall = {f"Recall@{k}": 0.0 for k in k_values}
    precision = {f"P@{k}": 0.0 for k in k_values}
    mrr = {f"MRR@{k}": 0.0 for k in k_values}

    cutoffs = ",".join(str(k) for k in k_values)
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels,
        {
            f"map_cut.{cutoffs}",
            f"ndcg_cut.{cutoffs}",
            f"recall.{cutoffs}",
            f"P.{cutoffs}",
        },
    )
    scores = evaluator.evaluate(results)

    for query_scores in scores.values():
        for k in k_values:
            ndcg[f"NDCG@{k}"] += query_scores[f"ndcg_cut_{k}"]
            map_[f"MAP@{k}"] += query_scores[f"map_cut_{k}"]
            recall[f"Recall@{k}"] += query_scores[f"recall_{k}"]
            precision[f"P@{k}"] += query_scores[f"P_{k}"]

    k_max = max(k_values)
    for query_id, doc_scores in results.items():
        top_hits = sorted(
            doc_scores.items(), key=lambda item: item[1], reverse=True
        )[:k_max]
        relevant = {
            doc_id
            for doc_id, score in qrels.get(query_id, {}).items()
            if score > 0
        }
        for k in k_values:
            for rank, (doc_id, _) in enumerate(top_hits[:k]):
                if doc_id in relevant:
                    mrr[f"MRR@{k}"] += 1.0 / (rank + 1)
                    break

    query_count = len(scores) or 1
    qrel_count = len(qrels) or 1
    for k in k_values:
        ndcg[f"NDCG@{k}"] = round(ndcg[f"NDCG@{k}"] / query_count, 5)
        map_[f"MAP@{k}"] = round(map_[f"MAP@{k}"] / query_count, 5)
        recall[f"Recall@{k}"] = round(recall[f"Recall@{k}"] / query_count, 5)
        precision[f"P@{k}"] = round(
            precision[f"P@{k}"] / query_count, 5
        )
        mrr[f"MRR@{k}"] = round(mrr[f"MRR@{k}"] / qrel_count, 5)

    return {
        "ndcg": ndcg,
        "map": map_,
        "recall": recall,
        "precision": precision,
        "mrr": mrr,
    }


channel_scores: dict[str, dict[str, dict[str, dict[str, float]]]] = {
    "hybrid+code": {},
    "hybrid": {},
    "semantic": {},
    "lexical": {},
    "code": {},
}
task_results: dict[str, dict[str, dict[str, dict[str, float]]]] = {
    task.name: {} for task in tasks
}

for task in tasks:
    chunks, semantic_index, lexical_index = task_indexes[task.name]

    hybrid_code_results = run_hybrid_retrieval(
        task, chunks, semantic_index, lexical_index, embedding_session, with_code=True
    )
    channel_scores["hybrid+code"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], hybrid_code_results
    )
    task_results[task.name]["hybrid+code"] = hybrid_code_results

    headline = channel_scores["hybrid+code"][task.name]
    print(
        f"{task.name}: NDCG@10={headline['ndcg']['NDCG@10']}, "
        f"MRR@10={headline['mrr']['MRR@10']}, "
        f"Recall@10={headline['recall']['Recall@10']}"
    )


humaneval: NDCG@10=0.99303, MRR@10=0.99695, Recall@10=0.9939
mbpp: NDCG@10=0.66361, MRR@10=0.84073, Recall@10=0.70323


## 10. Ablations: what's each piece contributing

`semantic`, `lexical`, `code` alone, and `hybrid` without the code
channel, are not alternatives to section 9's result -- QAChat never
runs any of them by themselves in `RAGSession._retrieve`. They exist to
explain *why* `hybrid+code` scored what it did: which channel is doing
the work, and where a weak channel might be dragging the fusion down
instead of helping it. Same indexes as section 9, no rebuilding.


In [22]:
for task in tasks:
    chunks, semantic_index, lexical_index = task_indexes[task.name]

    semantic_results = run_semantic_retrieval(task, chunks, semantic_index, embedding_session)
    channel_scores["semantic"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], semantic_results
    )
    task_results[task.name]["semantic"] = semantic_results

    lexical_results = run_lexical_retrieval(task, chunks, lexical_index)
    channel_scores["lexical"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], lexical_results
    )
    task_results[task.name]["lexical"] = lexical_results

    code_results = run_code_retrieval(task, chunks)
    channel_scores["code"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], code_results
    )
    task_results[task.name]["code"] = code_results

    hybrid_results = run_hybrid_retrieval(
        task, chunks, semantic_index, lexical_index, embedding_session, with_code=False
    )
    channel_scores["hybrid"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], hybrid_results
    )
    task_results[task.name]["hybrid"] = hybrid_results


In [23]:
HEADLINE_MEASURES = (
    ("mrr", "MRR@10"),
    ("ndcg", "NDCG@10"),
    ("recall", "Recall@10"),
    ("recall", "Recall@100"),
)

header = ["task", "channel", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>12}" for h in header))

for task in tasks:
    for channel in ("hybrid+code", "hybrid", "semantic", "lexical", "code"):
        scores = channel_scores[channel][task.name]
        values = [
            f"{scores[group][label]:.3f}" for group, label in HEADLINE_MEASURES
        ]
        row = [task.name, channel, *values]
        print(" | ".join(f"{cell:>12}" for cell in row))


        task |      channel |       MRR@10 |      NDCG@10 |    Recall@10 |   Recall@100
   humaneval |  hybrid+code |        0.997 |        0.993 |        0.994 |        1.000
   humaneval |       hybrid |        0.997 |        0.967 |        0.970 |        1.000
   humaneval |     semantic |        1.000 |        0.954 |        0.948 |        1.000
   humaneval |      lexical |        0.997 |        0.971 |        0.979 |        1.000
   humaneval |         code |        0.909 |        0.911 |        0.924 |        0.924
        mbpp |  hybrid+code |        0.841 |        0.664 |        0.703 |        0.992
        mbpp |       hybrid |        0.999 |        0.698 |        0.663 |        0.992
        mbpp |     semantic |        0.999 |        0.942 |        0.974 |        0.995
        mbpp |      lexical |        0.999 |        0.637 |        0.555 |        0.707
        mbpp |         code |        0.177 |        0.176 |        0.186 |        0.186


**Splitting.** `documents_split_into_multiple_chunks`: 31/164 HumanEval
documents split, 500/500 MBPP documents split. That gap traces to how
each task's corpus text is built, not to anything about the code itself:
HumanEval's docstring lives inside the function body (`ast` sees one
`FunctionDef`), while MBPP's task description sits as a `#` comment
*before* `def` (`extract_source_units` always carves that into a
separate `python_module_context` unit). 478/500 MBPP documents split
into exactly `(python_module_context, python_function)`; the remaining
22 are genuinely multi-function or multi-class solutions, a different
and smaller effect. Propagation means every MBPP query now has (at
least) two gold chunks instead of one -- a comment chunk that's
near-verbatim the query text, and a code chunk that usually isn't --
which is why MRR@10 (first-hit rank) stays high across channels while
Recall@10 (found *every* gold chunk) drops sharply for anything but
semantic search.

**The code-aware channel.** The hypothesis in section 6 held: `code`
alone scores 0.909 MRR@10 on HumanEval (queries literally contain the
function name) vs. 0.177 on MBPP (queries are independent descriptions).
Its effect on `hybrid_search` fusion is asymmetric, though. On
HumanEval, adding `code` closes real distance to the ceiling (NDCG@10
.967 -> .993, Recall@10 .970 -> .994). On MBPP, `hybrid` was already
*worse* than semantic alone before `code` was added (NDCG@10 .942 ->
.698, driven by a weak lexical channel), and `code` doesn't recover it
-- MRR@10 drops further, .999 -> .841. A retrieval channel that's weak
on its own can still drag down RRF fusion, not just fail to help it.

Semantic search alone is close to solving both tasks (MRR@10 ~1.0
on both), so there was little headroom for any channel to move the
aggregate numbers -- but the code-aware channel and real chunking each
surfaced a real, task-specific mechanism (name-literal matching;
comment-vs-code corpus structure).

### A worked example

The aggregate scores above show whether the code-aware channel helps
*on average*; this picks one HumanEval query -- where `code_search`'s
literal name matching has the best chance of working, since the query
text is the function's own signature -- and one MBPP query -- where it
does not, since the query is an independently-written description -- and
shows what each channel actually returned for both.


In [24]:
def show_worked_example(task: BenchmarkTask):
    results = task_results[task.name]
    lexical_results = results["lexical"]

    example_query_id = None
    for query_id, doc_scores in lexical_results.items():
        gold_ids = {
            doc_id
            for doc_id, relevance in qrels_chunks[task.name][query_id].items()
            if relevance > 0
        }
        ranked_ids = [
            doc_id
            for doc_id, _ in sorted(
                doc_scores.items(), key=lambda item: item[1], reverse=True
            )
        ]
        if ranked_ids and not (set(ranked_ids[:1]) & gold_ids):
            example_query_id = query_id
            break
    if example_query_id is None:
        example_query_id = next(iter(task.queries))

    gold_ids = set(qrels_chunks[task.name][example_query_id])

    print(f"=== {task.name} ===")
    print("Query:")
    print(task.queries[example_query_id])
    print(f"\nGold chunk id(s): {sorted(gold_ids)}")

    for channel_name in ("hybrid+code", "hybrid", "semantic", "lexical", "code"):
        ranked = sorted(
            results[channel_name].get(example_query_id, {}).items(),
            key=lambda item: item[1],
            reverse=True,
        )
        gold_rank = next(
            (rank for rank, (doc_id, _) in enumerate(ranked, start=1) if doc_id in gold_ids),
            None,
        )
        print(f"--- {channel_name} (gold chunk ranked #{gold_rank}) ---")
        for rank, (doc_id, score) in enumerate(ranked[:5], start=1):
            marker = " <-- gold" if doc_id in gold_ids else ""
            print(f"  {rank}. {doc_id}  score={score:.4f}{marker}")
        print()


show_worked_example(humaneval_task)
show_worked_example(mbpp_task)


=== humaneval ===
Query:


def correct_bracketing(brackets: str):
    """ brackets is a string of "<" and ">".
    return True if every opening bracket has a corresponding closing bracket.

    >>> correct_bracketing("<")
    False
    >>> correct_bracketing("<>")
    True
    >>> correct_bracketing("<<><>>")
    True
    >>> correct_bracketing("><<>")
    False
    """


Gold chunk id(s): ['5566ad4e6c6e19fe']
--- hybrid+code (gold chunk ranked #2) ---
  1. 2b7d8db8af27d083  score=0.0489
  2. 5566ad4e6c6e19fe  score=0.0487 <-- gold
  3. 785b70c78667670c  score=0.0317
  4. 19417e4fa980c847  score=0.0294
  5. 572654cdcd7c9722  score=0.0292

--- hybrid (gold chunk ranked #2) ---
  1. 2b7d8db8af27d083  score=0.0325
  2. 5566ad4e6c6e19fe  score=0.0325 <-- gold
  3. 785b70c78667670c  score=0.0317
  4. 19417e4fa980c847  score=0.0294
  5. 572654cdcd7c9722  score=0.0292

--- semantic (gold chunk ranked #1) ---
  1. 5566ad4e6c6e19fe  score=0.9121 <-- gold
  2. 2b7d8db8af27d083  score=0.8459
  3.

## 11. Save results

In [25]:
import json
from datetime import datetime, timezone

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

run_record = {
    "benchmark": "codeRAG-bench (canonical: humaneval, mbpp) -- QAChat-native pipeline",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "embedding_model": MODEL_ID,
    "backend": selected_backend,
    "chunking_profile": {
        "target_tokens": profile.target_tokens,
        "max_tokens": profile.max_tokens,
        "overlap_tokens": profile.overlap_tokens,
    },
    "documents_split_into_multiple_chunks": split_counts,
    "code_graph_entities": len(code_graph.entities),
    "code_graph_relations": len(code_graph.relations),
    "scores": channel_scores,
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_path = results_dir / f"{stamp}_canonical_qachat_native.json"
output_path.write_text(json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")


Saved: benchmarks/results/20260919T114327Z_canonical_qachat_native.json


## 12. Release the model

In [27]:
embedding_session.close()

print("Embedding model released.")


Embedding model released.
